# Overview of pylandstats

In [ ]:
import swisslandstats as sls

import pylandstats as pls

The data used in this notebook ships with the docs in the `data` directory, namely:
- the land use/land cover (LULC) data (see [A03-swisslandstats-preprocessing.ipynb](https://github.com/martibosch/pylandstats-notebooks/blob/main/notebooks/A03-swisslandstats-preprocessing.ipynb) for how it is derived from the raw SLS data).
- the elevation zones vector data (see [A04-elevation-zones.ipynb](https://github.com/martibosch/pylandstats-notebooks/blob/main/notebooks/A04-elevation-zones.ipynb) for more details).

## Landscape analysis

We can load landscapes from raster files and compute pandas data frames of patch, class and landscape level. See the notebook [01-landscape-analysis.ipynb](https://github.com/martibosch/pylandstats-notebooks/blob/main/notebooks/01-landscape-analysis.ipynb) for more thorough demonstration.

In [ ]:
URBAN_CLASS_VAL = 1
input_filepath = "data/veveyse/LU18_4.tif"

In [ ]:
ls = pls.Landscape(input_filepath)
ls.plot_landscape(cmap=sls.noas04_4_cmap, norm=sls.noas04_4_norm, legend=True)

In [ ]:
patch_metrics_df = ls.compute_patch_metrics_df()
patch_metrics_df.head()

In [ ]:
class_metrics_df = ls.compute_class_metrics_df()
class_metrics_df

In [ ]:
landscape_metrics_df = ls.compute_landscape_metrics_df()
landscape_metrics_df

(spatiotemporal-analysis)=
## Spatio-temporal analysis

Given a temporally-ordered sequence of landscape snapshots, we can also analyze the spatio-temporal patterns of landscape change. To that end, pylandstats can compute pandas dataframes with the evolution of the metrics and plot them, both at the class and landscape level. See the notebook [02-spatiotemporal-analysis.ipynb](https://github.com/martibosch/pylandstats-notebooks/blob/main/notebooks/02-spatiotemporal-analysis.ipynb) for a more thorough demonstration.

In [ ]:
input_filepaths = [
    "data/veveyse/LU85_4.tif",
    "data/veveyse/LU97_4.tif",
    "data/veveyse/LU09_4.tif",
    "data/veveyse/LU18_4.tif",
]
years = ["1980", "1992", "2004", "2013"]

In [ ]:
sta = pls.SpatioTemporalAnalysis(input_filepaths, dates=years)

In [ ]:
sta.compute_class_metrics_df()

In [ ]:
sta.compute_landscape_metrics_df()

We can also plot the time series of metrics at the class level, e.g., the evolution of the proportion of landscape occupied by the land use class value `1` (urban):

In [ ]:
sta.plot_metric("proportion_of_landscape", class_val=URBAN_CLASS_VAL)

or we can also plot at the landscape level by not providing any `class_val` argument, e.g., the evolution of the area-weighted mean fractal dimension of all the patches of the landscape:

In [ ]:
sta.plot_metric("fractal_dimension_am")

(zonal-analysis)=
## Zonal analysis

Zonal analysis is a common procedure to compute statistics for a set of specified spatial zones. PyLandStats features three classes to perform zonal analysis, `ZonalAnalysis`, `BufferAnalysis` and `ZonalGridAnalysis`. The first allows user to fully customize how the zones are defined, while `BufferAnalysis` and `ZonalGriAnalysis` provide a convenient way to instantiate specific cases of zonal analysis, i.e., adding buffers around a feature of interest or as a regular rectangular grid over the landscape, respectively. See the notebook [03-zonal-analysis.ipynb](https://github.com/martibosch/pylandstats-notebooks/blob/main/notebooks/03-zonal-analysis.ipynb) for a thorough demonstration of the use cases described above.

To define the zones of a `ZonalAnalysis`, we can use - among other options - any geographic data file that can be read by [geopandas.read_file](https://geopandas.org/en/stable/docs/reference/api/geopandas.read_file.html#geopandas.read_file). For instance, we can use a geopackage file defining three elevation zones in our landscape:

In [ ]:
elev_zones_filepath = "data/elev-zones.gpkg"

za = pls.ZonalAnalysis(input_filepath, elev_zones_filepath, zone_index="elev-zone")
# plot the landscapes of each zone
fig = za.plot_landscapes(
    cmap=sls.noas04_4_cmap, show_kwargs=dict(norm=sls.noas04_4_norm)
)

Analogously to the spatio-temporal analysis, we can use the `compute_class_metrics_df` and `compute_landscape_metrics_df` methods to compute the metrics for each zone:

In [ ]:
za.compute_class_metrics_df()

In [ ]:
za.compute_landscape_metrics_df()

We can also use the `plot_metric` method to plot the metrics computed for each zone, e.g., how the proportion of landscape occupied by the land use class value `1` (urban) changes across elevation zones

In [ ]:
za.plot_metric("proportion_of_landscape", class_val=URBAN_CLASS_VAL)

Like in the spatio-temporal analysis, the plots at the landscape level can obtained by not providing any `class_val` argument.

In order to visualize such information in space, the zonal statistics can be computed in the form of a geo-data frame with the `compute_zonal_statistics_gdf` method as in:

In [ ]:
metrics = ["proportion_of_landscape", "edge_density"]
zonal_statistics_gdf = za.compute_zonal_statistics_gdf(
    metrics=metrics, class_val=URBAN_CLASS_VAL
)

zonal_statistics_gdf.head()

the computed metrics are essentially the same as those obtained using the `compute_class_metrics_df` or `compute_landscape_metrics_df` (depending on whether a `class_val` argument is provided or not), with an additional column featuring the vector geometry of each zone. This actually corresponds to a geopandas geo-data frame, and as such, we can use [its `geopandas.GeoDataFrame.explore` method](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.explore.html) to obtain an interactive map as in:

In [ ]:
zonal_statistics_gdf.explore()

## Spatio-temporal zonal analysis

We might also be interested in performing the same zonal analysis at different points in time. This is why pylandstats features an additional `SpatioTemporalZonalAnalysis` analysis class - as well as `SpatioTemporalBufferAnalysis` and `SpatioTemporalZonalGridAnalysis`. See the notebook [04-spatiotemporal-zonal-analysis.ipynb](https://github.com/martibosch/pylandstats-notebooks/blob/main/notebooks/04-spatiotemporal-zonal-analysis.ipynb) for a more thorough demonstration.

Let us take the sequence of landscapes `input_filepaths` from [the spatio-temporal analysis above](#spatiotemporal-analysis) and let us use again the `dates` argument to specify the dates that correspond to each landscape.
Let us also take the latitude and longitude of the center of Lausanne as well as the elevation zones from [the zonal analysis above](#zonal-analysis). Now we can construct our `SpatioTemporalZonalAnalysis` instance and evaluate the sensitive of our spatio-temporal analysis to the extent of the map:

In [ ]:
stza = pls.SpatioTemporalZonalAnalysis(
    input_filepaths, elev_zones_filepath, dates=years, zone_index="elev-zone"
)

In [ ]:
stza.compute_class_metrics_df()

In [ ]:
stza.plot_metric("proportion_of_landscape", class_val=URBAN_CLASS_VAL)